## Getting the Raw Data

Reading the files and putting the data into a palatable format.

In [1]:
import pandas as pd
import numpy as np
import sklearn as sk

In [2]:
!pwd

/home/ericwang/vae_cloud_computing/scripts


In [128]:
# Put path to two datasets to compare here.
raw_path = '../data/raw/appraise_labeled.csv'
syn_path = '../data/syn/appraise_dp/dpvae.csv'

raw_df = pd.read_csv(raw_path)
syn_df = pd.read_csv(syn_path)

In [129]:
shared_cols = list(set(raw_df.columns) & set(syn_df.columns))

raw_df = raw_df[shared_cols]
syn_df = syn_df[shared_cols]

In [130]:
# Since all values are integers, round the float values.
syn_df = syn_df.round(0)
syn_df

,IN_BYTES,ANOMALY,PROTOCOL,TOTAL_FLOWS_EXP,IN_PKTS,L4_SRC_PORT,OUT_PKTS,OUT_BYTES,L4_DST_PORT
0,105.0,1.0,11,4707830.0,7.0,27015,0.0,5646.0,45509
1,8558.0,1.0,202,47999.0,18597.0,41186,0.0,-0.0,47303
2,173.0,1.0,11,2548928.0,22.0,19999,6.0,-0.0,623
3,845.0,1.0,11,530964.0,88.0,46270,0.0,-0.0,46270
4,1647.0,1.0,11,18698.0,27.0,46270,198.0,-0.0,47303
...,...,...,...,...,...,...,...,...,...
14242187,15237.0,0.0,11,456310.0,9.0,129,6.0,0.0,623
14242188,106649.0,1.0,11,10856.0,23.0,41249,2.0,-0.0,45509
14242189,161.0,0.0,202,4406329.0,86.0,19999,45.0,139.0,623
14242190,607.0,1.0,202,614861.0,13380.0,21001,34.0,760.0,46270


## Preprocessing

Creating train-test split and applying some basic data transforms.

In [5]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

In [131]:
# Train test split (raw data)
raw_X = raw_df.loc[:, raw_df.columns != 'ANOMALY'].to_numpy()
raw_y = raw_df['ANOMALY'].to_numpy()

X_train, X_test, y_train, y_test = train_test_split(raw_X, raw_y, train_size=0.8)
X_train

array([[    246,       6, 2050393, ...,       6,     240,     443],
       [  68040,      17,   76416, ...,    1628,   85116,   47808],
       [   6126,       6,  315329, ...,       4,    1096,     443],
       ...,
       [     88,       6,   47751, ...,       2,      80,    4446],
       [   7920,       6, 4004875, ...,     110,    4400,   49944],
       [   2040,      17, 2476428, ...,       0,       0,    1947]])

In [132]:
# Standardize (raw data)
raw_scaler = StandardScaler()
X_train = raw_scaler.fit_transform(X_train)
X_train

array([[-0.09137997, -0.85275934,  0.13415654, ..., -0.14537954,
        -0.0314277 , -0.72585979],
       [ 0.18883674,  0.64098086, -1.39884806, ...,  0.45540997,
         0.04003043,  1.48845537],
       [-0.06707583, -0.85275934, -1.21330652, ..., -0.14612034,
        -0.03070702, -0.72585979],
       ...,
       [-0.09203304, -0.85275934, -1.4211095 , ..., -0.14686114,
        -0.0315624 , -0.53871942],
       [-0.05966059, -0.85275934,  1.65202118, ..., -0.10685789,
        -0.02792534,  1.58831344],
       [-0.08396473,  0.64098086,  0.46501835, ..., -0.14760194,
        -0.03162976, -0.65554775]])

In [133]:
# Train test split (syn data)
syn_X = syn_df.loc[:, syn_df.columns != 'ANOMALY'].to_numpy()
syn_y = syn_df['ANOMALY'].to_numpy()

X_train_syn, X_test_syn, y_train_syn, y_test_syn = train_test_split(syn_X, syn_y, train_size=0.8)
X_train_syn

array([[ 3.662110e+05,  2.470000e+02,  5.283060e+05, ..., -0.000000e+00,
         2.118100e+04,  5.905000e+03],
       [ 1.211954e+06,  1.100000e+01,  4.427082e+06, ...,  0.000000e+00,
         6.876000e+03,  4.627000e+04],
       [ 1.125300e+04,  2.020000e+02,  6.234730e+05, ...,  0.000000e+00,
         5.010000e+02,  4.982800e+04],
       ...,
       [ 4.963000e+04,  2.470000e+02,  1.688470e+05, ...,  2.100000e+01,
         1.430000e+03,  4.627000e+04],
       [ 1.140000e+02,  2.020000e+02,  2.457910e+05, ...,  2.000000e+00,
         1.470000e+02,  4.730300e+04],
       [ 9.200000e+01,  1.100000e+01,  8.463100e+04, ...,  4.000000e+00,
         1.981670e+05,  4.730300e+04]])

In [134]:
# Standardize (syn data)
syn_scaler = StandardScaler()
X_train_syn = syn_scaler.fit_transform(X_train_syn)
X_train_syn

array([[ 0.71071808,  1.26078056, -0.4542177 , ..., -0.30659272,
        -0.16258207, -0.98404383],
       [ 3.06278683, -1.03336817,  2.23110907, ..., -0.30659272,
        -0.16943354,  0.85570907],
       [-0.27644425,  0.82333695, -0.38867033, ..., -0.30659272,
        -0.17248689,  1.01787532],
       ...,
       [-0.1697152 ,  1.26078056, -0.70179921, ..., -0.3029159 ,
        -0.17204194,  0.85570907],
       [-0.30742256,  0.82333695, -0.64880315, ..., -0.30624255,
        -0.17265644,  0.90279106],
       [-0.30748374, -1.03336817, -0.75980395, ..., -0.30589237,
        -0.07781346,  0.90279106]])

# Evaluation

Evaluating using a bunch of different ML/statistical methods.

In [12]:
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
from sklearn.svm import SVC, LinearSVC
from sklearn.naive_bayes import GaussianNB
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, precision_score, recall_score, f1_score, accuracy_score
from sklearn.neighbors import KNeighborsClassifier as KNN
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis as QDA

In [135]:
results_summary = {
    'logreg': {},
    'xgboost': {},
    'mlp': {},
    'svm': {},
    'nb': {},
    'qda': {}
}
results_summary

{'logreg': {}, 'xgboost': {}, 'mlp': {}, 'svm': {}, 'nb': {}, 'qda': {}}

## Logistic Regression

Not looking good lol.

In [12]:
# Train on raw data
logreg = LogisticRegression()
logreg.fit(X_train, y_train)

LogisticRegression()

In [13]:
# Evaluate on raw data test split.
scaled_X_test = raw_scaler.transform(X_test)

y_pred_train = logreg.predict(X_train)
y_pred_test = logreg.predict(scaled_X_test)
                                     
train_score = {
    'accuracy': accuracy_score(y_train, y_pred_train),
    'precision': precision_score(y_train, y_pred_train),
    'recall': recall_score(y_train, y_pred_train)
}
test_score = {
    'accuracy': accuracy_score(y_test, y_pred_test),
    'precision': precision_score(y_test, y_pred_test),
    'recall': recall_score(y_test, y_pred_test)
}

print(f'TRAIN DATA SUMMARY (REAL):\n {train_score}')
print(f'TEST DATA SUMMARY (REAL):\n {test_score}')

results_summary['logreg']['real'] = test_score
results_summary

TRAIN DATA SUMMARY (REAL):
 {'accuracy': 0.9513724054257165, 'precision': 0.877155370828641, 'recall': 0.9645053657614038}
TEST DATA SUMMARY (REAL):
 {'accuracy': 0.9514116018883103, 'precision': 0.8772836734801418, 'recall': 0.9647061141378355}


{'logreg': {'real': {'accuracy': 0.9514116018883103,
   'precision': 0.8772836734801418,
   'recall': 0.9647061141378355}},
 'xgboost': {},
 'mlp': {}}

In [14]:
# Train on syn data.
logreg_syn = LogisticRegression()
logreg_syn.fit(X_train_syn, y_train_syn)

LogisticRegression()

In [15]:
# Evaluate on raw data test split (syn trained model).
scaled_X_test = syn_scaler.transform(X_test)

y_pred_train = logreg_syn.predict(X_train_syn)
y_pred_test = logreg_syn.predict(scaled_X_test)
                                     
train_score = {
    'accuracy': accuracy_score(y_train_syn, y_pred_train),
    'precision': precision_score(y_train_syn, y_pred_train),
    'recall': recall_score(y_train_syn, y_pred_train)
}
test_score = {
    'accuracy': accuracy_score(y_test, y_pred_test),
    'precision': precision_score(y_test, y_pred_test),
    'recall': recall_score(y_test, y_pred_test)
}

print(f'TRAIN DATA SUMMARY (SYN):\n {train_score}')
print(f'TEST DATA SUMMARY (REAL):\n {test_score}')

results_summary['logreg']['syn'] = test_score
results_summary

TRAIN DATA SUMMARY (SYN):
 {'accuracy': 0.9640752843314704, 'precision': 0.9548893997036355, 'recall': 0.9516902819362189}
TEST DATA SUMMARY (REAL):
 {'accuracy': 0.3386938217113341, 'precision': 0.3011484751566174, 'recall': 0.9972707328690682}


{'logreg': {'real': {'accuracy': 0.9514116018883103,
   'precision': 0.8772836734801418,
   'recall': 0.9647061141378355},
  'syn': {'accuracy': 0.3386938217113341,
   'precision': 0.3011484751566174,
   'recall': 0.9972707328690682}},
 'xgboost': {},
 'mlp': {}}

## Gradient Boosted Trees

Using XGBoost for easy parallelism.

In [16]:
# Train on raw data.
gbt = XGBClassifier(
    n_estimators=100, 
    learning_rate=1.0,
    max_depth=3, 
    objective='binary:logistic'
).fit(X_train, y_train)
gbt

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              gamma=None, grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=1.0, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=3, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=100, n_jobs=None,
              num_parallel_tree=None, random_state=None, ...)

In [17]:
# Evaluate on raw data test split.
scaled_X_test = raw_scaler.transform(X_test)

y_pred_train = gbt.predict(X_train)
y_pred_test = gbt.predict(scaled_X_test)
                                     
train_score = {
    'accuracy': accuracy_score(y_train, y_pred_train),
    'precision': precision_score(y_train, y_pred_train),
    'recall': recall_score(y_train, y_pred_train)
}
test_score = {
    'accuracy': accuracy_score(y_test, y_pred_test),
    'precision': precision_score(y_test, y_pred_test),
    'recall': recall_score(y_test, y_pred_test)
}

print(f'TRAIN DATA SUMMARY (REAL):\n {train_score}')
print(f'TEST DATA SUMMARY (REAL):\n {test_score}')

results_summary['xgboost']['real'] = test_score
results_summary

TRAIN DATA SUMMARY (REAL):
 {'accuracy': 0.9986693049028325, 'precision': 0.9965478829301667, 'recall': 0.9987921706902988}
TEST DATA SUMMARY (REAL):
 {'accuracy': 0.9986977512807486, 'precision': 0.9966302789213192, 'recall': 0.9988144202654169}


{'logreg': {'real': {'accuracy': 0.9514116018883103,
   'precision': 0.8772836734801418,
   'recall': 0.9647061141378355},
  'syn': {'accuracy': 0.3386938217113341,
   'precision': 0.3011484751566174,
   'recall': 0.9972707328690682}},
 'xgboost': {'real': {'accuracy': 0.9986977512807486,
   'precision': 0.9966302789213192,
   'recall': 0.9988144202654169}},
 'mlp': {}}

In [18]:
# Train on syn data.
gbt_syn = XGBClassifier(
    n_estimators=100, 
    learning_rate=1.0,
    max_depth=3, 
    objective='binary:logistic'
).fit(X_train_syn, y_train_syn)
gbt_syn

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              gamma=None, grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=1.0, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=3, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=100, n_jobs=None,
              num_parallel_tree=None, random_state=None, ...)

In [19]:
# Evaluate on raw data test split (syn trained model).
scaled_X_test = syn_scaler.transform(X_test)

y_pred_train = gbt_syn.predict(X_train_syn)
y_pred_test = gbt_syn.predict(scaled_X_test)
                                     
train_score = {
    'accuracy': accuracy_score(y_train_syn, y_pred_train),
    'precision': precision_score(y_train_syn, y_pred_train),
    'recall': recall_score(y_train_syn, y_pred_train)
}
test_score = {
    'accuracy': accuracy_score(y_test, y_pred_test),
    'precision': precision_score(y_test, y_pred_test),
    'recall': recall_score(y_test, y_pred_test)
}

print(f'TRAIN DATA SUMMARY (SYN):\n {train_score}')
print(f'TEST DATA SUMMARY (REAL):\n {test_score}')

results_summary['xgboost']['syn'] = test_score
results_summary

TRAIN DATA SUMMARY (SYN):
 {'accuracy': 0.9790616466086626, 'precision': 0.9746633883418309, 'recall': 0.9708771964476406}
TEST DATA SUMMARY (REAL):
 {'accuracy': 0.41425203226216184, 'precision': 0.3008794363817625, 'recall': 0.7950221876090113}


{'logreg': {'real': {'accuracy': 0.9514116018883103,
   'precision': 0.8772836734801418,
   'recall': 0.9647061141378355},
  'syn': {'accuracy': 0.3386938217113341,
   'precision': 0.3011484751566174,
   'recall': 0.9972707328690682}},
 'xgboost': {'real': {'accuracy': 0.9986977512807486,
   'precision': 0.9966302789213192,
   'recall': 0.9988144202654169},
  'syn': {'accuracy': 0.41425203226216184,
   'precision': 0.3008794363817625,
   'recall': 0.7950221876090113}},
 'mlp': {}}

## MLP Classifier

Scuffed MLP classifier.

In [20]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from tqdm import tqdm

class ClassifierNet(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, 4 * input_dim)
        self.fc2 = nn.Linear(4 * input_dim, 2 * input_dim)
        self.fc3 = nn.Linear(2 * input_dim, 1)

    def forward(self, X):
        out1 = F.relu(self.fc1(X))
        out2 = F.relu(self.fc2(out1))
        out3 = F.relu(self.fc3(out2))
        out4 = F.sigmoid(out3)
        return out4

def mlp_train(X, y, epochs=10, batch_size=256, device='cuda'):
    """Train a simple feed forward network with given input features and labels."""
    
    mlp = ClassifierNet(X.shape[1]).to(device)
    optimizer = torch.optim.Adam(mlp.parameters(), lr=0.005)

    print(mlp)

    dtype = torch.get_default_dtype()

    X_t = torch.from_numpy(X).to(dtype)
    y_t = torch.from_numpy(y).to(dtype)
    dataset = TensorDataset(X_t, y_t)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, pin_memory=True)

    for epoch in range(epochs):
        total_loss = 0
        best_loss = float('inf')
        worst_loss = float('-inf')

        pbar = tqdm(loader, desc=f'Epoch {epoch}')
        for i, (X, y) in enumerate(pbar):
            optimizer.zero_grad()
            X = X.to(device)
            y = y.to(device).reshape(-1, 1)
            y_hat = mlp(X)
            loss = F.binary_cross_entropy(y_hat, y)
            
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

            if i % 300 == 0:
                pbar.set_description(f'Epoch {epoch} (loss={loss.item()})', refresh=True)
            
            best_loss = min(loss.item(), best_loss)
            worst_loss = max(loss.item(), worst_loss)

            
        log_text = (
            f'Average Minibatch Loss: {total_loss / len(loader)}\n'
            f'Best Minibatch Loss: {best_loss}\n'
            f'Worst Minibatch Loss: {worst_loss}'
        )
        print(log_text)

    return mlp

def generate_preds(mlp, X, batch_size=256, device='cuda'):
    """Generate predictions for the given X."""

    dtype = torch.get_default_dtype()

    X_t = torch.from_numpy(X).to(dtype)
    y_t = torch.arange(len(X_t))
    dataset = TensorDataset(X_t, y_t)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, pin_memory=True)

    all_preds = []

    with torch.inference_mode():
        total_correct = 0
        for X, _ in tqdm(loader, desc=f'Inference ({device})'):
            X = X.to(device)
            probs = mlp(X)
            preds = torch.where(probs > 0.5, 1.0, 0.0)
            all_preds.append(preds)
    results = torch.concat(all_preds).cpu().numpy()
    return results

In [21]:
# Train on raw data.
batch_size = 1024
DEVICE = 'cuda:2'
mlp = mlp_train(X_train, y_train, epochs=2, batch_size=batch_size, device=DEVICE)

ClassifierNet(
  (fc1): Linear(in_features=8, out_features=32, bias=True)
  (fc2): Linear(in_features=32, out_features=16, bias=True)
  (fc3): Linear(in_features=16, out_features=1, bias=True)
)


Epoch 0 (loss=0.5588821768760681): 100%|██████████████████████████████████████████████████████████████| 11810/11810 [01:05<00:00, 179.18it/s]


Average Minibatch Loss: 0.5611953389634529
Best Minibatch Loss: 0.5199493169784546
Worst Minibatch Loss: 0.7331439852714539


Epoch 1 (loss=0.5239282250404358): 100%|██████████████████████████████████████████████████████████████| 11810/11810 [01:05<00:00, 180.83it/s]

Average Minibatch Loss: 0.533616428535233
Best Minibatch Loss: 0.4856536388397217
Worst Minibatch Loss: 0.6727590560913086


In [22]:
# Evaluate on raw data test split.
batch_size=8192 * 4

scaled_X_test = raw_scaler.transform(X_test)

y_pred_train = generate_preds(mlp, X_train, batch_size=batch_size, device=DEVICE)
y_pred_test = generate_preds(mlp, scaled_X_test, batch_size=batch_size, device=DEVICE)

Inference (cuda:2): 100%|████████████████████████████████████████████████████████████████████████████████████| 93/93 [00:20<00:00,  4.46it/s]


In [23]:
train_score = {
    'accuracy': accuracy_score(y_train, y_pred_train),
    'precision': precision_score(y_train, y_pred_train),
    'recall': recall_score(y_train, y_pred_train)
}
test_score = {
    'accuracy': accuracy_score(y_test, y_pred_test),
    'precision': precision_score(y_test, y_pred_test),
    'recall': recall_score(y_test, y_pred_test)
}

print(f'TRAIN DATA SUMMARY (REAL):\n {train_score}')
print(f'TEST DATA SUMMARY (REAL):\n {test_score}')

results_summary['mlp']['real'] = test_score
results_summary

TRAIN DATA SUMMARY (REAL):
 {'accuracy': 0.9720126506996486, 'precision': 0.9660454242658085, 'recall': 0.9346800456097696}
TEST DATA SUMMARY (REAL):
 {'accuracy': 0.9722250889114695, 'precision': 0.9665222396450378, 'recall': 0.935073574320088}


{'logreg': {'real': {'accuracy': 0.9514116018883103,
   'precision': 0.8772836734801418,
   'recall': 0.9647061141378355},
  'syn': {'accuracy': 0.3386938217113341,
   'precision': 0.3011484751566174,
   'recall': 0.9972707328690682}},
 'xgboost': {'real': {'accuracy': 0.9986977512807486,
   'precision': 0.9966302789213192,
   'recall': 0.9988144202654169},
  'syn': {'accuracy': 0.41425203226216184,
   'precision': 0.3008794363817625,
   'recall': 0.7950221876090113}},
 'mlp': {'real': {'accuracy': 0.9722250889114695,
   'precision': 0.9665222396450378,
   'recall': 0.935073574320088}}}

In [24]:
# Train on syn data.
batch_size = 1024
mlp_syn = mlp_train(X_train_syn, y_train_syn, epochs=2, batch_size=batch_size, device=DEVICE)

ClassifierNet(
  (fc1): Linear(in_features=8, out_features=32, bias=True)
  (fc2): Linear(in_features=32, out_features=16, bias=True)
  (fc3): Linear(in_features=16, out_features=1, bias=True)
)


Epoch 0 (loss=0.6931471824645996): 100%|██████████████████████████████████████████████████████████████| 11810/11810 [01:05<00:00, 179.57it/s]


Average Minibatch Loss: 0.6931471839383131
Best Minibatch Loss: 0.6931471824645996
Worst Minibatch Loss: 0.6931626796722412


Epoch 1 (loss=0.6931471824645996): 100%|██████████████████████████████████████████████████████████████| 11810/11810 [01:04<00:00, 181.79it/s]

Average Minibatch Loss: 0.6931471824645996
Best Minibatch Loss: 0.6931471824645996
Worst Minibatch Loss: 0.6931471824645996


In [25]:
# Evaluate on raw data test split.
batch_size=8192 * 4

scaled_X_test = syn_scaler.transform(X_test)

y_pred_train = generate_preds(mlp_syn, X_train_syn, batch_size=batch_size, device=DEVICE)
y_pred_test = generate_preds(mlp_syn, scaled_X_test, batch_size=batch_size, device=DEVICE)

Inference (cuda:2): 100%|████████████████████████████████████████████████████████████████████████████████████| 93/93 [00:19<00:00,  4.87it/s]


In [37]:
train_score = {
    'accuracy': accuracy_score(y_train_syn, y_pred_train),
    'precision': precision_score(y_train_syn, y_pred_train),
    'recall': recall_score(y_train_syn, y_pred_train)
}
test_score = {
    'accuracy': accuracy_score(y_test, y_pred_test),
    'precision': precision_score(y_test, y_pred_test),
    'recall': recall_score(y_test, y_pred_test)
}

print(f'TRAIN DATA SUMMARY (SYN):\n {train_score}')
print(f'TEST DATA SUMMARY (REAL):\n {test_score}')

results_summary['mlp']['syn'] = test_score
results_summary

TRAIN DATA SUMMARY (SYN):
 {'accuracy': 0.9753237594733054, 'precision': 0.9742720346055577, 'recall': 0.9613432843828972}
TEST DATA SUMMARY (REAL):
 {'accuracy': 0.714042124454888, 'precision': 0.0, 'recall': 0.0}


{'logreg': {'real': {'accuracy': 0.9515624338456328,
   'precision': 0.877723742285522,
   'recall': 0.9645720832236586},
  'syn': {'accuracy': 0.5075277054490029,
   'precision': 0.3627238252093681,
   'recall': 0.959699628594189}},
 'xgboost': {'real': {'accuracy': 0.8221992225538761,
   'precision': 0.7867885152528488,
   'recall': 0.5167283357239515},
  'syn': {'accuracy': 0.7147426992040307, 'precision': 0.0, 'recall': 0.0}},
 'mlp': {'real': {'accuracy': 0.9848218727507515,
   'precision': 0.9800560654912668,
   'recall': 0.9664586809585818},
  'syn': {'accuracy': 0.714042124454888, 'precision': 0.0, 'recall': 0.0}}}

In [83]:
y_pred_train.sum()

0.0

## SVM

In [78]:
from sklearn.kernel_approximation import RBFSampler

In [83]:
# Train on raw data
rbf = RBFSampler(n_components=50)
X_train_rbf = rbf.fit_transform(X_train)

svm = LinearSVC()
svm.fit(X_train_rbf, y_train)

LinearSVC()

In [84]:
# Evaluate on raw data test split.
scaled_X_test = raw_scaler.transform(X_test)

y_pred_train = svm.predict(X_train_rbf)
y_pred_test = svm.predict(rbf.transform(scaled_X_test))
                                     
train_score = {
    'accuracy': accuracy_score(y_train, y_pred_train),
    'precision': precision_score(y_train, y_pred_train),
    'recall': recall_score(y_train, y_pred_train)
}
test_score = {
    'accuracy': accuracy_score(y_test, y_pred_test),
    'precision': precision_score(y_test, y_pred_test),
    'recall': recall_score(y_test, y_pred_test)
}

print(f'TRAIN DATA SUMMARY (REAL):\n {train_score}')
print(f'TEST DATA SUMMARY (REAL):\n {test_score}')

results_summary['svm']['real'] = test_score
results_summary

TRAIN DATA SUMMARY (REAL):
 {'accuracy': 0.9112520144004826, 'precision': 0.8555160500302204, 'recall': 0.8287378661047826}
TEST DATA SUMMARY (REAL):
 {'accuracy': 0.9113326400355646, 'precision': 0.8553778547938516, 'recall': 0.8292137386337006}


{'logreg': {},
 'xgboost': {},
 'mlp': {},
 'svm': {'real': {'accuracy': 0.9113326400355646,
   'precision': 0.8553778547938516,
   'recall': 0.8292137386337006}},
 'nb': {},
 'qda': {}}

In [136]:
# Train on syn data
rbf_syn = RBFSampler(n_components=50)
X_train_syn_rbf = rbf_syn.fit_transform(X_train_syn)

svm = LinearSVC()
svm.fit(X_train_syn_rbf, y_train_syn)

LinearSVC()

In [138]:
# Evaluate on raw data test split (syn trained model).
scaled_X_test = raw_scaler.transform(X_test)

y_pred_train = svm.predict(X_train_syn_rbf)
y_pred_test = svm.predict(rbf_syn.transform(scaled_X_test))
                                     
train_score = {
    'accuracy': accuracy_score(y_train, y_pred_train),
    'precision': precision_score(y_train, y_pred_train),
    'recall': recall_score(y_train, y_pred_train)
}
test_score = {
    'accuracy': accuracy_score(y_test, y_pred_test),
    'precision': precision_score(y_test, y_pred_test),
    'recall': recall_score(y_test, y_pred_test)
}

print(f'TRAIN DATA SUMMARY (REAL):\n {train_score}')
print(f'TEST DATA SUMMARY (REAL):\n {test_score}')

results_summary['svm']['syn'] = test_score
results_summary

TRAIN DATA SUMMARY (REAL):
 {'accuracy': 0.5661619749989415, 'precision': 0.28516009791869, 'recall': 0.3459036267197554}
TEST DATA SUMMARY (REAL):
 {'accuracy': 0.7228538861721495, 'precision': 0.8013500232762634, 'recall': 0.0359796604894906}


{'logreg': {},
 'xgboost': {},
 'mlp': {},
 'svm': {'syn': {'accuracy': 0.7228538861721495,
   'precision': 0.8013500232762634,
   'recall': 0.0359796604894906}},
 'nb': {},
 'qda': {}}

## Naive Bayes

In [75]:
# Train on raw data
nb = GaussianNB()
nb.fit(X_train, y_train)

GaussianNB()

In [76]:
# Evaluate on raw data test split.
scaled_X_test = raw_scaler.transform(X_test)

y_pred_train = nb.predict(X_train)
y_pred_test = nb.predict(scaled_X_test)
                                     
train_score = {
    'accuracy': accuracy_score(y_train, y_pred_train),
    'precision': precision_score(y_train, y_pred_train),
    'recall': recall_score(y_train, y_pred_train)
}
test_score = {
    'accuracy': accuracy_score(y_test, y_pred_test),
    'precision': precision_score(y_test, y_pred_test),
    'recall': recall_score(y_test, y_pred_test)
}

print(f'TRAIN DATA SUMMARY (REAL):\n {train_score}')
print(f'TEST DATA SUMMARY (REAL):\n {test_score}')

results_summary['nb']['real'] = test_score
results_summary

TRAIN DATA SUMMARY (REAL):
 {'accuracy': 0.9033736081121131, 'precision': 0.7501724214733473, 'recall': 0.9913023709919356}
TEST DATA SUMMARY (REAL):
 {'accuracy': 0.9035264247004531, 'precision': 0.7503295583785992, 'recall': 0.9914405240590205}


{'logreg': {},
 'xgboost': {},
 'mlp': {},
 'svm': {},
 'nb': {'real': {'accuracy': 0.9035264247004531,
   'precision': 0.7503295583785992,
   'recall': 0.9914405240590205}}}

In [77]:
# Train on syn data
nb_syn = GaussianNB()
nb_syn.fit(X_train_syn, y_train_syn)

GaussianNB()

In [78]:
# Evaluate on raw data test split (syn trained model).
scaled_X_test = syn_scaler.transform(X_test)

y_pred_train = nb_syn.predict(X_train_syn)
y_pred_test = nb_syn.predict(scaled_X_test)
                                     
train_score = {
    'accuracy': accuracy_score(y_train_syn, y_pred_train),
    'precision': precision_score(y_train_syn, y_pred_train),
    'recall': recall_score(y_train_syn, y_pred_train)
}
test_score = {
    'accuracy': accuracy_score(y_test, y_pred_test),
    'precision': precision_score(y_test, y_pred_test),
    'recall': recall_score(y_test, y_pred_test)
}

print(f'TRAIN DATA SUMMARY (SYN):\n {train_score}')
print(f'TEST DATA SUMMARY (REAL):\n {test_score}')

results_summary['nb']['syn'] = test_score
results_summary

TRAIN DATA SUMMARY (SYN):
 {'accuracy': 0.8036651669471612, 'precision': 0.42002145737334823, 'recall': 0.8949769446021003}
TEST DATA SUMMARY (REAL):
 {'accuracy': 0.842589983170329, 'precision': 0.6454833382349614, 'recall': 0.9933088478476562}


{'logreg': {},
 'xgboost': {},
 'mlp': {},
 'svm': {},
 'nb': {'real': {'accuracy': 0.9035264247004531,
   'precision': 0.7503295583785992,
   'recall': 0.9914405240590205},
  'syn': {'accuracy': 0.842589983170329,
   'precision': 0.6454833382349614,
   'recall': 0.9933088478476562}}}

## QDA

In [62]:
# Train on raw data
qda = QDA()
qda.fit(X_train, y_train)

QuadraticDiscriminantAnalysis()

In [63]:
# Evaluate on raw data test split.
scaled_X_test = raw_scaler.transform(X_test)

y_pred_train = qda.predict(X_train)
y_pred_test = qda.predict(scaled_X_test)
                                     
train_score = {
    'accuracy': accuracy_score(y_train, y_pred_train),
    'precision': precision_score(y_train, y_pred_train),
    'recall': recall_score(y_train, y_pred_train)
}
test_score = {
    'accuracy': accuracy_score(y_test, y_pred_test),
    'precision': precision_score(y_test, y_pred_test),
    'recall': recall_score(y_test, y_pred_test)
}

print(f'TRAIN DATA SUMMARY (REAL):\n {train_score}')
print(f'TEST DATA SUMMARY (REAL):\n {test_score}')

results_summary['qda']['real'] = test_score
results_summary

TRAIN DATA SUMMARY (REAL):
 {'accuracy': 0.9227383971855286, 'precision': 0.7913677502136994, 'recall': 0.9900881973835115}
TEST DATA SUMMARY (REAL):
 {'accuracy': 0.9225454745120454, 'precision': 0.7908815727766976, 'recall': 0.9900993315627462}


{'logreg': {},
 'xgboost': {},
 'mlp': {},
 'svm': {},
 'nb': {},
 'qda': {'real': {'accuracy': 0.9225454745120454,
   'precision': 0.7908815727766976,
   'recall': 0.9900993315627462}}}

In [64]:
# Train on syn data
qda_syn = QDA()
qda_syn.fit(X_train_syn, y_train_syn)

QuadraticDiscriminantAnalysis()

In [65]:
# Evaluate on raw data test split (syn trained model).
scaled_X_test = syn_scaler.transform(X_test)

y_pred_train = qda_syn.predict(X_train_syn)
y_pred_test = qda_syn.predict(scaled_X_test)
                                     
train_score = {
    'accuracy': accuracy_score(y_train_syn, y_pred_train),
    'precision': precision_score(y_train_syn, y_pred_train),
    'recall': recall_score(y_train_syn, y_pred_train)
}
test_score = {
    'accuracy': accuracy_score(y_test, y_pred_test),
    'precision': precision_score(y_test, y_pred_test),
    'recall': recall_score(y_test, y_pred_test)
}

print(f'TRAIN DATA SUMMARY (SYN):\n {train_score}')
print(f'TEST DATA SUMMARY (REAL):\n {test_score}')

results_summary['qda']['syn'] = test_score
results_summary

TRAIN DATA SUMMARY (SYN):
 {'accuracy': 0.5733126740591972, 'precision': 0.5116699414664481, 'recall': 0.08127634276233534}
TEST DATA SUMMARY (REAL):
 {'accuracy': 0.6982100612854059, 'precision': 3.9585147652600746e-05, 'recall': 2.3205598118490106e-06}


{'logreg': {},
 'xgboost': {},
 'mlp': {},
 'svm': {},
 'nb': {},
 'qda': {'real': {'accuracy': 0.9225454745120454,
   'precision': 0.7908815727766976,
   'recall': 0.9900993315627462},
  'syn': {'accuracy': 0.6982100612854059,
   'precision': 3.9585147652600746e-05,
   'recall': 2.3205598118490106e-06}}}

# Save and Load Output

In [20]:
import json
import os
import datetime

## Save New File

In [139]:
# Save results so far.
name = 'dpvae_svm2'
dataset = 'appraise'
timestamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
save_dir = './'

output_file = os.path.join(save_dir, f'{name}_{dataset}_{timestamp}.json')


with open(output_file, mode='w') as file:
    json.dump(results_summary, file, indent=2)
    print(f'Saved results at: {output_file}')

results_summary

Saved results at: ./dpvae_svm2_appraise_20250408_180153.json


{'logreg': {},
 'xgboost': {},
 'mlp': {},
 'svm': {'syn': {'accuracy': 0.7228538861721495,
   'precision': 0.8013500232762634,
   'recall': 0.0359796604894906}},
 'nb': {},
 'qda': {}}

## Modify Existing Files

In [27]:
# Load half results from elsewhere so we can fill out the rest.
load_file = '/home/ericwang/vae_cloud_computing/scripts/dpwgan_appraise_20250316_135704.json'

with open(load_file, mode='r') as file:
    loaded_results = json.load(file)

loaded_results

{'logreg': {'real': {'accuracy': 0.9505539105169567,
   'precision': 0.8752586321942833,
   'recall': 0.9643653541119126},
  'syn': {'accuracy': 0.6381213879715484,
   'precision': 0.3966309107832217,
   'recall': 0.5116277454238113}},
 'xgboost': {'real': {'accuracy': 0.9989798996570558,
   'precision': 0.9984131874546832,
   'recall': 0.9980155376225817},
  'syn': {'accuracy': 0.7143090573267284, 'precision': 0.0, 'recall': 0.0}},
 'mlp': {}}

In [31]:
out = loaded_results.copy()
out['mlp'] = results_summary['mlp']
out

{'logreg': {'real': {'accuracy': 0.9505539105169567,
   'precision': 0.8752586321942833,
   'recall': 0.9643653541119126},
  'syn': {'accuracy': 0.6381213879715484,
   'precision': 0.3966309107832217,
   'recall': 0.5116277454238113}},
 'xgboost': {'real': {'accuracy': 0.9989798996570558,
   'precision': 0.9984131874546832,
   'recall': 0.9980155376225817},
  'syn': {'accuracy': 0.7143090573267284, 'precision': 0.0, 'recall': 0.0}},
 'mlp': {'real': {'accuracy': 0.9857900419154071,
   'precision': 0.973700226700281,
   'recall': 0.97661687796459},
  'syn': {'accuracy': 0.7144397122020407, 'precision': 0.0, 'recall': 0.0}}}

In [32]:
# Save results so far.
name = 'dpwgan'
dataset = 'appraise'
timestamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
save_dir = './'

output_file = os.path.join(save_dir, f'{name}_{dataset}_{timestamp}.json')

with open(output_file, mode='w') as file:
    json.dump(out, file, indent=2)
    print(f'Saved results at: {output_file}')

out

Saved results at: ./dpwgan_appraise_20250316_143929.json


{'logreg': {'real': {'accuracy': 0.9505539105169567,
   'precision': 0.8752586321942833,
   'recall': 0.9643653541119126},
  'syn': {'accuracy': 0.6381213879715484,
   'precision': 0.3966309107832217,
   'recall': 0.5116277454238113}},
 'xgboost': {'real': {'accuracy': 0.9989798996570558,
   'precision': 0.9984131874546832,
   'recall': 0.9980155376225817},
  'syn': {'accuracy': 0.7143090573267284, 'precision': 0.0, 'recall': 0.0}},
 'mlp': {'real': {'accuracy': 0.9857900419154071,
   'precision': 0.973700226700281,
   'recall': 0.97661687796459},
  'syn': {'accuracy': 0.7144397122020407, 'precision': 0.0, 'recall': 0.0}}}